In [ ]:
# Breast Cancer Wisconsin Dataset: Complete Analysis

# This Jupyter notebook provides a comprehensive analysis using the Wisconsin Breast Cancer dataset.
# Steps include:
# - Data loading and exploration
# - Visualizations for understanding data distribution and feature relationships
# - Preprocessing, scaling, and train/test split
# - Model training: Logistic Regression and Random Forest
# - Evaluation using confusion matrix, classification report, and ROC curves
# - Feature importance visualization

# Each stage contains clear explanations in Markdown cells.

# --- 

# 1. Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# 2. Load and Explore Data
# Markdown
"""
## Data Loading and Exploration

Let's load the Wisconsin Breast Cancer dataset and explore its basic properties.
"""

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target


# Show dataset information
display(df.head())
display(df.describe())
print(f"Target classes: {data.target_names}")
sns.countplot(x='target', data=df)
plt.title('Target Variable Distribution')
plt.show()

# 3. Visualizations
# Markdown
"""
## Visualizations

Let's visualize the features to gain some preliminary insights.
"""

# Visualization 1: Distribution of a prominent feature
plt.figure(figsize=(6,4))
sns.histplot(df['mean radius'], bins=30, kde=True)
plt.title('Distribution of Mean Radius')
plt.show()

# Visualization 2: Pairplot for selected features
sns.pairplot(df[['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'target']], hue='target', diag_kind='kde')
plt.suptitle('Pairplot of Selected Features', y=1.02)
plt.show()

# Visualization 3: Correlation heatmap
plt.figure(figsize=(12,10))
sns.heatmap(df.drop('target', axis=1).corr(), cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

# Visualization 4: Boxplot of a feature by target class
plt.figure(figsize=(7,5))
sns.boxplot(x='target', y='mean symmetry', data=df)
plt.title('Mean Symmetry by Diagnosis')
plt.show()

# 4. Preprocessing and Scaling
# Markdown
"""
## Preprocessing and Scaling

We scale features for better model performance.
"""

X = df.drop('target', axis=1)
y = df['target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Train/Test Split
# Markdown
"""
## Train/Test Split

We split the data into training and test sets.
"""

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 6. Model Training: Logistic Regression and Random Forest
# Markdown
"""
## Model Training

We will train two models: Logistic Regression and Random Forest Classifier.
"""

# Logistic Regression
logreg = LogisticRegression(max_iter=10000, random_state=42)
logreg.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 7. Evaluation: Confusion Matrix, Classification Report, ROC Curve
# Markdown
"""
## Model Evaluation

We'll compare both models using confusion matrices, classification reports, and ROC curves.
"""

for model, name in zip([logreg, rf], ["Logistic Regression", "Random Forest"]):
    print("Model:", name)
    print("Accuracy:", model.score(X_test, y_test))
    print(f"### {name}")
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    print(classification_report(y_test, y_pred))

    # ROC curve
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)
    fpr, tpr, thresholds = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

# 8. Feature Importance (Random Forest)
# Markdown
"""
## Feature Importance

Random Forests allow us to estimate feature importances.
"""

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:10]  # Top 10 features

plt.figure(figsize=(8, 6))
sns.barplot(x=importances[indices], y=np.array(data.feature_names)[indices])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf, X_scaled, y, cv=5)
print("Cross-validation accuracy:", scores.mean())

## Key Insight

False negatives (predicting benign when malignant) are more critical in this context. Therefore, recall for the malignant class is prioritized alongside overall accuracy.
